In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

# Pull every main_stats CSV referenced by one or more experiment manifests and
# tag each row with the SF / BW of its run so we can group later.
exp_paths = [
    'analysis/data/experiments/exp2_run_20260605_164813.json',
    'analysis/data/experiments/exp2_run_20260607_055702.json',
]
runs = []
for p in exp_paths:
    manifest = pd.read_json(p)
    runs.extend(list(manifest["experiments"]))

frames = []
for run in runs:
    try:
        d = pd.read_csv(run["main_stats"])
    except (FileNotFoundError, pd.errors.EmptyDataError):
        continue
    d["sf"] = run.get("sf")
    d["bw"] = run.get("bw")
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['node_time_us', 'node_bytes'])
df = df[df['node_bytes'] > 0]

print(f"Loaded {len(df)} valid rows across SF values: {sorted(df['sf'].dropna().unique())}")
df.groupby('sf').size()


In [ ]:
# Per-SF linear regression of SPI time on byte count.
# Slope ≈ µs / byte transferred; intercept ≈ fixed per-transaction overhead.
print(f"{'SF':>4}  {'n':>6}  {'slope (us/byte)':>16}  {'intercept (us)':>15}  {'R²':>7}  {'p-value':>11}")
print('-' * 72)
per_sf_results = {}
for sf, g in df.groupby('sf'):
    reg = linregress(g['node_bytes'], g['node_time_us'])
    per_sf_results[sf] = reg
    print(
        f"{int(sf):>4}  {len(g):>6}  {reg.slope:>16.3f}  {reg.intercept:>15.2f}  "
        f"{reg.rvalue**2:>7.4f}  {reg.pvalue:>11.2e}"
    )


In [ ]:
import matplotlib.pyplot as plt

sfs = sorted(df['sf'].dropna().unique())
cmap = plt.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(9, 6))
for i, sf in enumerate(sfs):
    g = df[df['sf'] == sf]
    reg = per_sf_results[sf]
    color = cmap(i / max(len(sfs) - 1, 1))
    ax.scatter(
        g['node_bytes'], g['node_time_us'],
        s=8, alpha=0.25, color=color,
    )
    xs = np.array([g['node_bytes'].min(), g['node_bytes'].max()])
    ax.plot(
        xs, reg.intercept + reg.slope * xs,
        color=color, linewidth=2,
        label=f"SF{int(sf)}  slope={reg.slope:.2f}  b={reg.intercept:.0f}  R²={reg.rvalue**2:.2f}",
    )

ax.set_xlabel('node_bytes')
ax.set_ylabel('node_time_us')
ax.set_title('SPI time vs bytes per SF (linear regression)')
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
